In [1]:
from google.colab import userdata
API_KEY = userdata.get('YOUTUBE_API_KEY')

In [2]:
!pip -q install google-api-python-client

import re, time, random
import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from difflib import SequenceMatcher
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata, files

youtube = build("youtube", "v3", developerKey=userdata.get("YOUTUBE_API_KEY"))

def run(call):
    """Run an API call. On failure print only the status (never the URL, so the key can't leak)."""
    try:
        return call.execute()
    except HttpError as e:
        print("   API error", e.resp.status, getattr(e, "reason", ""))
        return None

t = run(youtube.channels().list(part="snippet", forHandle="@YouTube"))
print("Key OK ->", t["items"][0]["snippet"]["title"] if t else "FAILED")

Key OK -> YouTube


In [3]:
!pip -q install google-api-python-client
import re, time, requests
import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata, files


# ---------------- settings ----------------
PER_CHANNEL       = 20     # raise up to this if the total is below the target
WINDOW_MONTHS     = 24      # look back about 2 years
MIN_AGE_DAYS      = 30      # skip videos younger than 30 days (views not settled)
MIN_SUBS          = 50_000
ALLOW_SHORTS      = False   # False = skip videos of 3 minutes or less


youtube = build("youtube", "v3", developerKey=userdata.get("YOUTUBE_API_KEY"))


def run(call):
    try:
        return call.execute()
    except HttpError as e:                       # never print the URL (it contains the key)
        print("   API error", e.resp.status, getattr(e, "reason", ""))
        return None


# ---------------- your channels ----------------
HANDLES = ["BeardMeatsFood","MattStonie","RandySantel","FuriousPete","ErikTheElectric",
 "LeahShutkever","RainaHuang","ZermattNeo","WakeNBite","FoodieWe","NickDiGiovanni",
 "JoshuaWeissman","GugaFoods","BayashiTV_","SortedFood","JOLLY","TryGuys","GoodMythicalMorning",
 "Sidemen","BetaSquad","MrBeast","DudePerfect","HowRidiculous","Airrack","StokesTwins",
 "RyanTrahan","MarkRober","MichelleKhare","YesTheory","MaxFosh","Niko","Ludwig","Jesser",
 "ZachKing","TheBucketListFamily",
 # the last five had no handle in your list, so these are guesses; unresolved ones are just skipped
 "KatinaEatsKilos","MaxvsFood","JoelHansen","FreakEating","TheFoodieWe"]


pool, seen = [], set()
for h in HANDLES:
    r = run(youtube.channels().list(part="snippet,statistics", forHandle=h))   # 1 quota unit
    time.sleep(0.1)
    if not (r and r.get("items")):
        print("not found:", h); continue
    it = r["items"][0]; st = it["statistics"]
    subs = int(st.get("subscriberCount", 0))
    if it["id"] in seen: continue
    seen.add(it["id"])
    if subs < MIN_SUBS:
        print("too small:", it["snippet"]["title"], subs); continue
    pool.append({"channel_id": it["id"], "channel_name": it["snippet"]["title"],
                 "subscribers": subs, "channel_video_count": int(st.get("videoCount", 0))})
print(len(pool), "channels ready")


# ---------------- helpers ----------------
now = datetime.now(timezone.utc)
end = now - timedelta(days=MIN_AGE_DAYS)
since = end - timedelta(days=30.4 * WINDOW_MONTHS)


def iso_to_sec(d):
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", d or "")
    if not m: return 0
    h, mi, s = (int(x) if x else 0 for x in m.groups()); return h*3600 + mi*60 + s


def get_uploads(cid, max_pages=30):
    pl, out, tok = "UU" + cid[2:], [], None
    for _ in range(max_pages):
        r = run(youtube.playlistItems().list(part="contentDetails,snippet", playlistId=pl,
                                             maxResults=50, pageToken=tok))
        if not r: break
        stop = False
        for it in r["items"]:
            if it["snippet"].get("title") in ("Private video", "Deleted video"): continue
            ts = it["contentDetails"].get("videoPublishedAt") or it["snippet"].get("publishedAt")
            if not ts: continue
            pub = datetime.fromisoformat(ts.replace("Z", "+00:00"))
            if pub > end: continue
            if pub < since: stop = True; break
            out.append((it["contentDetails"]["videoId"], pub))
        tok = r.get("nextPageToken")
        if stop or not tok: break
    return out


def drop_shorts(ups):
    if ALLOW_SHORTS or not ups: return ups
    keep = []
    for i in range(0, len(ups), 50):
        chunk = ups[i:i+50]
        r = run(youtube.videos().list(part="contentDetails", id=",".join(v for v, _ in chunk)))
        dur = {x["id"]: iso_to_sec(x["contentDetails"].get("duration")) for x in (r or {}).get("items", [])}
        keep += [(v, p) for v, p in chunk if dur.get(v, 0) > 180]
    return keep


def pick_even(ups, k):
    """k videos spread evenly over time."""
    ups = sorted(ups, key=lambda x: x[1])
    if len(ups) <= k: return [v for v, _ in ups]
    idx = np.linspace(0, len(ups) - 1, k).round().astype(int)
    return [ups[i][0] for i in idx]


# ---------------- read uploads once ----------------
avail = {}
for ch in pool:
    avail[ch["channel_id"]] = drop_shorts(get_uploads(ch["channel_id"]))
    print(f"{ch['channel_name']}: {len(avail[ch['channel_id']])} usable videos in window")
    time.sleep(0.1)


selected = {}
for cid, u in avail.items():
    if PER_CHANNEL is None:
        selected[cid] = [v for v, _ in sorted(u, key=lambda x: x[1])]   # every usable video
    else:
        selected[cid] = pick_even(u, PER_CHANNEL)                       # even spread, or all if fewer
total = sum(len(v) for v in selected.values())
print(f"\n{total} videos selected from {len(selected)} channels")


# ---------------- fetch details for the selected videos ----------------
info = {c["channel_id"]: c for c in pool}
ids = [(cid, v) for cid, vs in selected.items() for v in vs]
rows = []
for i in range(0, len(ids), 50):
    chunk = ids[i:i+50]
    r = run(youtube.videos().list(part="snippet,statistics,contentDetails",
                                  id=",".join(v for _, v in chunk)))
    for v in (r or {}).get("items", []):
        c = info[v["snippet"]["channelId"]]
        th = v["snippet"]["thumbnails"]
        thumb = (th.get("maxres") or th.get("high") or th.get("medium") or th.get("default"))["url"]
        st = v["statistics"]
        pub = datetime.fromisoformat(v["snippet"]["publishedAt"].replace("Z", "+00:00"))
        rows.append({
            "channel_name":        c["channel_name"],
            "channel_id":          c["channel_id"],
            "video_id":            v["id"],
            "title":               v["snippet"]["title"],
            "thumbnail":           thumb,
            "published_at":        v["snippet"]["publishedAt"],
            "views":               int(st["viewCount"]) if "viewCount" in st else None,
            "likes":               int(st["likeCount"]) if "likeCount" in st else None,
            "comments":            int(st["commentCount"]) if "commentCount" in st else None,
            "duration_sec":        iso_to_sec(v["contentDetails"].get("duration")),
            "age_days":            (now - pub).days,
            "tags_count":          len(v["snippet"].get("tags", [])),
            "category_id":         v["snippet"].get("categoryId"),
            "subscribers":         c["subscribers"],
            "channel_video_count": c["channel_video_count"],
        })
    time.sleep(0.2)


final = pd.DataFrame(rows).drop_duplicates("video_id")


final = final.sort_values(["channel_name", "published_at"]).reset_index(drop=True)
final.to_csv("challenge_videos.csv", index=False, encoding="utf-8-sig")


print(f"\nChannels: {final['channel_id'].nunique()}   Videos: {len(final)}")
print("Missing views:", final["views"].isna().sum())
print(final["channel_name"].value_counts())
pd.set_option("display.max_rows", None); pd.set_option("display.max_colwidth", 50)
display(final)
files.download("challenge_videos.csv")



not found: RyanTrahan
not found: TheFoodieWe
38 channels ready
BeardMeatsFood: 106 usable videos in window
Matt Stonie: 14 usable videos in window
Randy Santel: 135 usable videos in window
Furious Pete: 1 usable videos in window
ErikTheElectric: 4 usable videos in window
Leah Shutkever: 85 usable videos in window
Raina Huang: 273 usable videos in window
Zermatt Neo: 200 usable videos in window
Wake'N'Bite: 313 usable videos in window
Foodie We: 98 usable videos in window
Nick DiGiovanni: 32 usable videos in window
Joshua Weissman: 55 usable videos in window
Guga Foods: 120 usable videos in window
Bayashi TV: 52 usable videos in window
Sorted Food: 354 usable videos in window
JOLLY: 114 usable videos in window
The Try Guys: 144 usable videos in window
Good Mythical Morning: 519 usable videos in window
Sidemen: 106 usable videos in window
Beta Squad: 43 usable videos in window
MrBeast: 48 usable videos in window
Dude Perfect: 40 usable videos in window
How Ridiculous: 51 usable videos in

,channel_name,channel_id,video_id,title,thumbnail,published_at,views,likes,comments,duration_sec,age_days,tags_count,category_id,subscribers,channel_video_count
0,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,FTMHc-swSMQ,GTA In Real Life Prank,https://i.ytimg.com/vi/FTMHc-swSMQ/maxresdefau...,2024-11-12T20:33:55Z,4427969,113347,2288.0,901,734,0,24,19700000,271
1,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,8btpqys1Rxk,I Got Arrested Sneaking Into Super Bowl 2025 (...,https://i.ytimg.com/vi/8btpqys1Rxk/maxresdefau...,2025-02-12T22:08:47Z,4216965,97680,2170.0,646,641,0,24,19700000,271
2,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,idfJEicYrpU,"$1 vs $10,000,000 Road Trip!",https://i.ytimg.com/vi/idfJEicYrpU/maxresdefau...,2025-03-22T22:34:38Z,12900057,165471,2310.0,962,603,0,24,19700000,271
3,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,HUZHDrckHSs,How Long Could You Secretly Live In A Grocery ...,https://i.ytimg.com/vi/HUZHDrckHSs/maxresdefau...,2025-04-20T20:55:21Z,24988221,320717,4821.0,1243,574,0,24,19700000,271
4,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,nh7ODfOcmLA,I Gave Scammers Instant Karma,https://i.ytimg.com/vi/nh7ODfOcmLA/maxresdefau...,2025-05-26T19:54:16Z,18208964,314034,7801.0,1591,539,0,24,19700000,271
5,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,t-oLGxzHyWQ,Drive Thru A.I. Robot Prank,https://i.ytimg.com/vi/t-oLGxzHyWQ/maxresdefau...,2025-06-30T17:14:17Z,15829796,228519,6099.0,1084,504,0,24,19700000,271
6,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,T-rCZGEWLgA,I Secretly Hid in Famous Livestreams Again ......,https://i.ytimg.com/vi/T-rCZGEWLgA/maxresdefau...,2025-07-31T14:50:43Z,7545869,148340,8093.0,1424,473,0,24,19700000,271
7,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,srj4cV0ASWY,I Secretly Hid In Kai Cenat's Livestream (FINALE),https://i.ytimg.com/vi/srj4cV0ASWY/maxresdefau...,2025-08-27T15:02:29Z,12315461,202350,8680.0,2132,446,0,24,19700000,271
8,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,usGm7LfK86w,Fake Arresting Youtubers Prank,https://i.ytimg.com/vi/usGm7LfK86w/maxresdefau...,2025-10-01T01:27:33Z,8713500,221332,4012.0,996,411,0,24,19700000,271
9,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,KeFTEPJUGjA,Airrack vs Jesse James West FULL FIGHT,https://i.ytimg.com/vi/KeFTEPJUGjA/maxresdefau...,2025-11-02T21:11:29Z,1585142,46306,2146.0,3006,378,0,24,19700000,271


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>